# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*I chose an indexed line chart because it compares growth relative to each continent's 1970 population, making very different continent sizes directly comparable. Africa is highlighted as the fastest-growing continent, while the remaining continents provide grey context.*

In [2]:
# Q1
ACCENT = '#D55E00'

continent_year = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
population_1970 = continent_year[continent_year['Year'] == 1970].set_index('Continent')['Population']
continent_year['Index (1970=100)'] = (
    continent_year['Population'] / continent_year['Continent'].map(population_1970) * 100
)

palette = {c: '#B0B0B0' for c in continent_year['Continent'].unique()}
palette['Africa'] = ACCENT

fig = px.line(
    continent_year, x='Year', y='Index (1970=100)', color='Continent',
    color_discrete_map=palette, template='plotly_white',
    title='Africa grew fastest: its population reached 390% of 1970 size by 2022'
)
fig.update_traces(line=dict(width=2))
fig.update_traces(line=dict(width=4), selector=dict(name='Africa'))

africa_2022 = continent_year[(continent_year['Continent'] == 'Africa') & (continent_year['Year'] == 2022)].iloc[0]
fig.add_annotation(
    x=africa_2022['Year'], y=africa_2022['Index (1970=100)'], text='Africa',
    showarrow=False, xanchor='left', xshift=6, font=dict(color=ACCENT, size=12)
)

fig.update_layout(
    showlegend=False, hovermode='x unified',
    margin=dict(l=60, r=110, t=80, b=50), yaxis_title=None, xaxis_title=None
)
fig.update_xaxes(showgrid=False, dtick=10)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** A **sorted horizontal bar chart**. The task ranks 15 named countries by one magnitude (people added); bars sorted by length make the order the message and give long country names room. Colour encodes one thing: blue = African country, grey = other, which answers how many are in Africa at a glance. Values are direct-labelled, so axis, gridlines and legend are removed.

In [3]:

# Q2
added = df.assign(Added=(df['2022 Population'] - df['1970 Population']) / 1e6)
top15 = added.nlargest(15, 'Added')[['Country', 'Continent', 'Added']].sort_values('Added')
top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other continents')
n_africa = int((top15['Continent'] == 'Africa').sum())
top_name, top_val = top15.iloc[-1]['Country'], top15.iloc[-1]['Added']

fig = px.bar(
    top15, x='Added', y='Country', orientation='h', color='Group',
    color_discrete_map={'Africa': '#0072B2', 'Other continents': '#B0B0B0'},
    text=top15['Added'].round(0).astype(int).astype(str) + 'M',
    hover_name='Country',
    hover_data={'Added': ':.1f', 'Group': False, 'Country': False},
    template='plotly_white',
    title=f'{n_africa} of the 15 countries that added the most people since 1970 are in Africa; {top_name} alone added {top_val:,.0f}M'
)
fig.update_traces(textposition='outside', cliponaxis=False, marker_line_width=0)
fig.add_annotation(
    x=1, y=0.04, xref='paper', yref='paper', showarrow=False, xanchor='right', align='right',
    text='<span style="color:#0072B2"><b>■ Africa</b></span>   <span style="color:#808080"><b>■ Other continents</b></span>',
    font=dict(size=12)
)
fig.update_layout(showlegend=False, xaxis_title=None, yaxis_title=None,
                  margin=dict(l=130, r=60, t=90, b=40), bargap=0.25)
fig.update_yaxes(categoryorder='array', categoryarray=list(top15['Country']), showgrid=False)
fig.update_xaxes(visible=False, showgrid=False)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** A **scatter plot on log-log axes**. The question is a relationship between two numeric variables (land area and 2022 population); each dot is a country. Log scales are needed because both variables span many orders of magnitude. Colour = continent (colour-blind-safe Okabe-Ito palette), showing whether the pattern differs by continent; only the outliers are labelled directly.

In [4]:
# Q3
palette_continent = {
    'Africa': '#0072B2', 'Asia': '#E69F00', 'Europe': '#56B4E9',
    'North America': '#CC79A7', 'Oceania': '#999999', 'South America': '#009E73'
}
r = np.corrcoef(np.log10(df['Area']), np.log10(df['2022 Population']))[0, 1]

fig = px.scatter(
    df, x='Area', y='2022 Population', color='Continent',
    color_discrete_map=palette_continent,
    category_orders={'Continent': list(palette_continent)},
    log_x=True, log_y=True, hover_name='Country',
    hover_data={'Area': ':,.0f', '2022 Population': ':,.0f', 'Continent': False},
    template='plotly_white',
    title=f'Bigger countries tend to have more people (log-log r = {r:.2f}), but Russia, Canada, Greenland and Australia fall far below the trend'
)
fig.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color='white')))

label_pos = {'China': 'top center', 'India': 'bottom center', 'Russia': 'bottom center',
             'Canada': 'bottom center', 'Greenland': 'top center', 'Australia': 'bottom center',
             'Bangladesh': 'top center', 'Singapore': 'top center'}
for name, pos in label_pos.items():
    row = df[df['Country'] == name].iloc[0]
    fig.add_scatter(x=[row['Area']], y=[row['2022 Population']], mode='text', text=[name],
                    textposition=pos, textfont=dict(size=11, color='#333333'),
                    showlegend=False, hoverinfo='skip')

fig.update_layout(
    legend=dict(title_text='', orientation='h', yanchor='bottom', y=1.0, xanchor='left', x=0),
    xaxis_title='Land area (km², log scale)', yaxis_title='Population 2022 (log scale)',
    margin=dict(l=70, r=40, t=120, b=60)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** A **box plot per continent**. The question asks for the spread, not just the average: a box shows median, interquartile range and outliers for each continent side by side. A log y-axis is needed because density runs from about 0.03 to over 23,000 people per km². Sorted by median; only the highest-median continent is accented, the rest are grey.

In [5]:
# Q4
medians = df.groupby('Continent')['Density'].median().sort_values(ascending=False)
order = list(medians.index)
top_cont, low_cont = medians.index[0], medians.index[-1]
palette_box = {c: ('#0072B2' if c == top_cont else '#B0B0B0') for c in order}

fig = px.box(
    df, x='Continent', y='Density', color='Continent',
    color_discrete_map=palette_box, category_orders={'Continent': order},
    log_y=True, points='outliers', hover_name='Country',
    hover_data={'Density': ':,.1f', 'Continent': False},
    template='plotly_white',
    title=f'Density spans orders of magnitude inside every continent; {top_cont} has the highest median ({medians.iloc[0]:.0f}/km²), {low_cont} the lowest ({medians.iloc[-1]:.0f}/km²)'
)
fig.update_traces(marker=dict(size=5, opacity=0.7), line=dict(width=1.5))
fig.update_layout(showlegend=False, xaxis_title=None, yaxis_title='People per km² (log scale)',
                  margin=dict(l=70, r=40, t=100, b=50))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** A **choropleth map**, because the question is about where countries grew or shrank (joined via ISO-3 `CCA3`). Percentage change is polarity data, so a **diverging scale** with a grey midpoint at exactly 0%: orange = shrank, blue = grew (colour-blind-safe, no red/green). Clipped at −40% / +120% so extreme cases like Qatar do not wash out the map; hover shows true values.

In [6]:
# Q5
d = df.copy()
d['Change (%)'] = (d['2022 Population'] / d['2000 Population'] - 1) * 100
lo, hi = -40, 120
d['Change (clipped)'] = d['Change (%)'].clip(lo, hi)
n_shrank = int((d['Change (%)'] < 0).sum())
n_shrank_eu = int(((d['Change (%)'] < 0) & (d['Continent'] == 'Europe')).sum())

zero_pos = (0 - lo) / (hi - lo)
scale = [[0, '#D55E00'], [zero_pos, '#EFEFEF'], [1, '#0072B2']]

fig = px.choropleth(
    d, locations='CCA3', locationmode='ISO-3', color='Change (clipped)',
    color_continuous_scale=scale, range_color=(lo, hi), hover_name='Country',
    hover_data={'Change (%)': ':.1f', 'Change (clipped)': False, 'CCA3': False},
    template='plotly_white',
    title=f'{n_shrank} countries shrank between 2000 and 2022, {n_shrank_eu} of them in Europe; Africa and the Gulf grew fastest'
)
fig.update_geos(showframe=False, showcoastlines=False, showcountries=True, countrycolor='#FFFFFF',
                showland=True, landcolor='#F5F5F5', projection_type='natural earth')
fig.update_layout(
    coloraxis_colorbar=dict(title='Change 2000–2022', thickness=12, len=0.6,
                            tickvals=[-40, 0, 40, 80, 120],
                            ticktext=['−40%', '0', '+40%', '+80%', '≥ +120%']),
    margin=dict(l=10, r=10, t=90, b=10)
)
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** A **treemap**: part-to-whole across a two-level hierarchy (continent → country) in one figure, with tile area proportional to 2022 population. Asia is the single accent colour because it is the headline; the other continents are neutral grey with white borders. Tiles show share of the world total.

In [7]:
# Q6
total = df['2022 Population'].sum()
asia_share = df.loc[df['Continent'] == 'Asia', '2022 Population'].sum() / total * 100
ci_share = df.loc[df['Country'].isin(['China', 'India']), '2022 Population'].sum() / total * 100
palette_tree = {c: ('#56B4E9' if c == 'Asia' else '#D4D4D4') for c in df['Continent'].unique()}

fig = px.treemap(
    df, path=['Continent', 'Country'], values='2022 Population',
    color='Continent', color_discrete_map=palette_tree, template='plotly_white',
    title=f"Asia holds {asia_share:.0f}% of the world's 8 billion people, and China and India alone make up {ci_share:.0f}%"
)
fig.update_traces(
    texttemplate='%{label}<br>%{percentRoot:.1%}',
    textfont=dict(color='#222222'),
    marker=dict(line=dict(width=1.5, color='white')),
    hovertemplate='<b>%{label}</b><br>Population 2022: %{value:,.0f}<br>Share of world: %{percentRoot:.2%}<extra></extra>'
)
fig.update_layout(margin=dict(l=10, r=10, t=80, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** A **100% stacked bar chart**, one bar per year: the question is about shares of a whole changing over time. Africa = blue (rising), Europe = orange (falling), Asia and the rest in greys as context. Africa and Europe are labelled only in 1970 and 2022, the two points being compared. Years are categories because they are unevenly spaced.

In [8]:
# Q7
focus = ['Africa', 'Asia', 'Europe']
d7 = df_long.copy()
d7['Group'] = np.where(d7['Continent'].isin(focus), d7['Continent'], 'Rest of the world')
shares = d7.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
shares['Share (%)'] = shares['Population'] / shares.groupby('Year')['Population'].transform('sum') * 100
shares['Year'] = shares['Year'].astype(str)
first_year, last_year = shares['Year'].min(), shares['Year'].max()
shares['Label'] = np.where(
    shares['Year'].isin([first_year, last_year]) & shares['Group'].isin(['Africa', 'Europe']),
    shares['Share (%)'].round(0).astype(int).astype(str) + '%', ''
)

def share_of(group, year):
    return shares[(shares['Group'] == group) & (shares['Year'] == year)]['Share (%)'].iloc[0]

order7 = ['Africa', 'Asia', 'Europe', 'Rest of the world']
palette7 = {'Africa': '#0072B2', 'Asia': '#9E9E9E', 'Europe': '#E69F00', 'Rest of the world': '#DCDCDC'}

fig = px.bar(
    shares, x='Year', y='Share (%)', color='Group', text='Label',
    color_discrete_map=palette7,
    category_orders={'Group': order7, 'Year': sorted(shares['Year'].unique())},
    hover_data={'Share (%)': ':.1f', 'Label': False}, template='plotly_white',
    title=(f"Africa's share of world population rose from {share_of('Africa', first_year):.0f}% to "
           f"{share_of('Africa', last_year):.0f}% while Europe's fell from {share_of('Europe', first_year):.0f}% to "
           f"{share_of('Europe', last_year):.0f}%")
)
fig.update_traces(textposition='inside', insidetextanchor='middle', marker_line_width=0)
fig.update_traces(textfont=dict(color='white', size=13), selector=dict(name='Africa'))
fig.update_traces(textfont=dict(color='#222222', size=13), selector=dict(name='Europe'))
fig.update_layout(
    barmode='stack', bargap=0.25,
    legend=dict(title_text='', orientation='h', yanchor='bottom', y=1.0, xanchor='left', x=0),
    xaxis_title=None, yaxis_title=None, margin=dict(l=50, r=40, t=110, b=40)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, ticksuffix='%', range=[0, 100])
fig.show()